# Week 9: Vector Databases

Same logic as `build_passage_index.py`. Indexes `data/sample/passages.json` (the same illustrative Apple/Microsoft/macro passages from Week 8's cosine-similarity exercise) into a real, persistent ChromaDB collection, embedding with the same `sentence-transformers` model Week 8 used by hand. No API key needed — everything here runs locally.

In [ ]:
import json
from pathlib import Path

from ai_finance_course.chunking import chunk_document
from ai_finance_course.vector_store import add_chunks, get_or_create_collection, query_collection

PASSAGES_PATH = Path("data/sample/passages.json")
PERSIST_PATH = Path("data/processed/chroma")
COLLECTION_NAME = "sample_passages"

passages = json.loads(PASSAGES_PATH.read_text(encoding="utf-8"))
len(passages)

## Chunk and Index Every Passage

In [ ]:
collection = get_or_create_collection(PERSIST_PATH, COLLECTION_NAME)

for passage in passages:
    metadata = {"ticker": passage["ticker"], "doc_type": passage["doc_type"]}
    chunks = chunk_document(passage["text"], metadata, chunk_size=500, overlap=50)
    add_chunks(collection, chunks)

print(f"Indexed {collection.count()} chunks from {len(passages)} passages.")

## Query Without a Filter

In [ ]:
query = "did the company beat earnings expectations?"

for result in query_collection(collection, query, n_results=3):
    print(f"[{result['distance']:.3f}] ({result['metadata']['ticker']}) {result['text']}")

## Query With a Metadata Filter

In [ ]:
for result in query_collection(collection, query, n_results=3, where={"ticker": "AAPL"}):
    print(f"[{result['distance']:.3f}] ({result['metadata']['ticker']}) {result['text']}")

## Confirm Persistence and Idempotent Re-Indexing

In [ ]:
reopened = get_or_create_collection(PERSIST_PATH, COLLECTION_NAME)
print(f"Count after reopening the collection: {reopened.count()}")

for passage in passages:
    metadata = {"ticker": passage["ticker"], "doc_type": passage["doc_type"]}
    chunks = chunk_document(passage["text"], metadata, chunk_size=500, overlap=50)
    add_chunks(reopened, chunks)

print(f"Count after re-indexing the same passages (should be unchanged): {reopened.count()}")